In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import sys, os
sys.path.append("../")
from pathlib import Path
import numpy as np
import torch

from src.optimization_src.knot_opti import KnotOpti
from src.geometry_src import geom
from src.geometry_src.rolliness import rolliness
from src.plot_utils import plotly_plot, plotly_knot

TORCH_DTYPE = torch.float64
morton_knot = geom.rotated_morton_knot_parametric

from matplotlib import pyplot as plt

# Add ipywidgets for interactive slider
from ipywidgets import interact, IntSlider

In [3]:
def plot(ko, hist):
    from mpl_toolkits.mplot3d import Axes3D
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.widgets import Slider

    obj_hist = np.array(hist[0])
    params = hist[2]
    pts = list(map(lambda x: ko.co.compute_curve_from_opt_params(x, closed_curve=True), params))
    knots = list(map(lambda x: ko.co.reconstruct_full_knot(x), pts))
    fig = plt.figure(figsize=(7, 14))
    knot_ax = fig.add_subplot(2, 1, 1, projection='3d')
    hist_ax = fig.add_subplot(2, 1, 2)

    # Adjust layout to make space for slider
    plt.subplots_adjust(bottom=0.15)

    # Add a matplotlib slider below the plots
    ax_slider = fig.add_axes([0.2, 0.05, 0.6, 0.03])
    slider = Slider(
        ax=ax_slider,
        label="Knot idx",
        valmin=0,
        valmax=len(knots)-1,
        valinit=len(knots)-1,
        valstep=1,
        color="lightblue"
    )

    def plot_idx(idx):
        knot_ax.clear()
        hist_ax.clear()

        knot_ax.set_title("Knot optimization (3D)")
        x, y, z = ko.stretched_knot.T
        knot_ax.plot(x, y, z, color="red", label="Stretched knot")
        x, y, z = ko.tdr1.T
        knot_ax.plot(x, y, z, color="green", label="TDR")
        x, y, z = knots[int(idx)].T
        knot_ax.plot(x, y, z, label="Projected knot")
        knot_ax.set_box_aspect([1,1,1])

        hist_ax.set_title("Optimization history")
        obj_hist_arr = np.array(hist[0])
        hist_ax.plot(obj_hist_arr[:, 1], label="Knot")
        hist_ax.plot(obj_hist_arr[:, 2], label="TDR")
        hist_ax.plot(obj_hist_arr[:, 3], label="Curvature")
        hist_ax.plot(obj_hist_arr[:, 0], label="Objective function")

        # Add vertical bar at current idx
        hist_ax.axvline(idx, color='black', linestyle='--', alpha=0.7)
        # Show current index value as text
        hist_ax.text(idx, hist_ax.get_ylim()[1]*0.95, f"idx={int(idx)}", color='black', ha='left', va='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

        knot_ax.legend()
        hist_ax.legend()
        fig.canvas.draw_idle()

    # Connect the slider to the plot update
    slider.on_changed(plot_idx)
    plot_idx(slider.val)

    plt.tight_layout()
    plt.show()


def run(a, p, n, n_tdr, curve_opt_params, heights=False):
    ko = KnotOpti(a=a, p=p, n=n, n_tdr=n_tdr, rotated=True, curve_opt_params=curve_opt_params, **curve_opt_params)

    projected_knot = ko.optimize()

    plotly_plot(ko, heights=heights)

    print(f"Rho: {rolliness(projected_knot.numpy())[0]}")
    
    return ko

def save_all(ko):
    ko.save(ko.base_knot, "base")
    ko.save(ko.stretched_knot, "stretched")
    ko.save(ko.projected_knot, "projected")
    ko.save(ko.tdr1, "tdr1")
    ko.save(ko.tdr2, "tdr2")

def save_hist(ko):
    knots = ko.knots_hist()
    dir_path = Path(ko.save_path) / Path(ko.name("hist")).stem
    dir_path.mkdir(parents=True, exist_ok=True)
    old_path = ko.save_path
    ko.save_path = dir_path
    for i, knot in enumerate(knots):
        ko.save(knot, f"step_{i}")
    ko.save(ko.tdr1, "tdr1")
    ko.save(ko.tdr2, "tdr2")
    ko.save_path = old_path


# Figure 2

In [ ]:
# knot params
a=0.75
p=3
# n=int(2e3)
n=2500
n_tdr = n+1

curve_opt_params = {
    'w_tdr': 4,
    'w_curvature': 1e4,
    'max_iter': 400,
    'factor_cps_to_pts': 32,
    'n_cps_int_per_seg': 9,
}

ko = run(a,p,n,n_tdr,curve_opt_params, heights=True)

In [ ]:
save_all(ko)

## Curve approximation vs. Smoothness

In [ ]:
# knot params
a=0.75
p=3
# n=int(2e3)
n=2000
n_tdr = n+1

curve_opt_params = {
    'w_tdr': 0,
    'w_curvature': 0,
    'max_iter': 400,
    'factor_cps_to_pts': 128,
    'n_cps_int_per_seg': 7,
    'curvature_damping': 0
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

In [ ]:
curve_opt_params = {
    'w_tdr': 1.0,
    'w_curvature': 1e5,
    'max_iter': 400,
    'factor_cps_to_pts': 512,
    'n_cps_int_per_seg': 7,
    'curvature_damping': 0,
    'tdr_damping': 2
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

# Figure 3

In [4]:
a = 0.9
p = 3
n = 1000
n_tdr = n+1

curve_opt_params = {
    'n_tdr_portion_indices': 40,
    # 'w_tdr': 4.5,
    'w_tdr': 6,
    'w_curvature': 1e6,
    'max_iter': 700,
    'factor_cps_to_pts': 16,
    'n_cps_int_per_seg': 7,
    'curvature_cps': 2,
    'tdr_damping': 2.0,  # Damping factor for the TDR attraction, must be even for symmetry preservation.
}

ko = run(a,p,n,n_tdr,curve_opt_params,heights=True)

Fixing 0 parameters.
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            3     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.06831D-01    |proj g|=  6.04808D-01

At iterate    1    f=  5.89828D-01    |proj g|=  2.79371D-01

At iterate    2    f=  5.47445D-01    |proj g|=  2.13179D-01

At iterate    3    f=  5.17845D-01    |proj g|=  8.38906D-01

At iterate    4    f=  4.86009D-01    |proj g|=  9.62221D-01

At iterate    5    f=  4.53252D-01    |proj g|=  9.15678D-01

At iterate    6    f=  4.41494D-01    |proj g|=  3.04504D-01

At iterate    7    f=  3.58260D-01    |proj g|=  4.46989D-01

At iterate    8    f=  2.46653D-01    |proj g|=  2.02711D+00

At iterate    9    f=  2.05786D-01    |proj g|=  1.77784D+00

At iterate   10    f=  1.64396D-01    |proj g|=  2.13196D+00

At iterate   11    f=  1.47251D-01    |proj g|=  8.28975D-02

At iterate   12    f=  1.42086D-01    |proj g|=  4.09445D-01

At it

    'data': [{'line': {'color': 'red'},
              'mode': 'lines',
         …

Weird hull, reordering failed. Reordered 2919/2920 faces.
Rho: 0.002232104817321614


In [18]:
save_hist(ko)

In [ ]:
dir_path = Path(ko.save_path) / Path(ko.name("hist")).stem
ko.save_path = str(dir_path)
knot = ko.load("step_1")

Scatter3d({
    'line': {'color': 'blue'},
    'mode': 'lines',
    'name': 'Knot',
    'x': array([0.32548528, 0.32181477, 0.31812388, ..., 0.34051458, 0.33541765,
                0.33040689]),
    'y': array([0.56416523, 0.567319  , 0.57045037, ..., 0.54391348, 0.5512168 ,
                0.55795636]),
    'z': array([0.56416523, 0.567319  , 0.57045037, ..., 0.58069476, 0.57401321,
                0.56843058])
})

In [16]:
save_all(ko)

In [ ]:
a = 0.9
p = 3
n = 1000
n_tdr = n+1

curve_opt_params = {
    'w_tdr': 7e-2,
    'w_curvature': 1000.0,
    'max_iter': 400,
    'factor_cps_to_pts': 32,
    'n_cps_int_per_seg': 7,
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

In [ ]:
a = 0.7
p = 3
n = 1000
n_tdr = n+1

curve_opt_params = {
    'w_tdr': 3e-2,
    'w_curvature': 1000.0,
    'max_iter': 400,
    'factor_cps_to_pts': 32,
    'n_cps_int_per_seg': 7,
    'curvature_damping': 0
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

In [ ]:
a = 0.5
p = 3
n = 2000
n_tdr = n+1

curve_opt_params = {
    'w_tdr': 1e1,
    'w_curvature': 1e5,
    'max_iter': 400,
    'factor_cps_to_pts': 28,
    'n_cps_int_per_seg': 7,
    'tdr_damping': 2
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

In [ ]:
a = 0.3
p = 3
n = 2000
n_tdr = n+1

curve_opt_params = {
    # 'w_tdr': 1e-2,
    'w_tdr': 10,
    # 'w_curvature': 100.0,
    'w_curvature': 10,
    'max_iter': 400,
    # 'factor_cps_to_pts': 32,
    'factor_cps_to_pts': 16,
    'n_cps_int_per_seg': 9,
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

# Generalized Knots

In [ ]:
# knot params
a=0.5
p=5
n=1001
n_tdr = n

# optimization params
curve_opt_params = {
    'w_tdr': 5.5e-1,
    'w_curvature': 5.5e7,
    # 'w_curvature': 1e2,
    'max_iter': 400,
    'factor_cps_to_pts': 64,
    'n_cps_int_per_seg': 17,
    'curvature_cps': 6,
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

In [ ]:
# knot params
a=0.5
p=7
n=2000
n_tdr = n+1

# optimization params
curve_opt_params = {
    'w_tdr': 4.5e-1,
    # 'w_curvature': 5.6e3,
    'w_curvature': 5e5,
    'curvature_cps': 2.5,
    'max_iter': 400,
    'factor_cps_to_pts': 14,
    'n_cps_int_per_seg': 15, 
    'tdr_damping': 2
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)

In [ ]:
# knot params
a=0.5
p=9
n=2001
n_tdr = n

# optimization params
curve_opt_params = {
    'w_tdr': 0.80,
    'w_curvature': 1e5,
    'max_iter': 1500,
    'curvature_cps': 8,
    'n_cps_int_per_seg': 25,
    'n_cps_ext_per_seg': 4,
    'n_tdr_portion_indices': 150,
    'factor_cps_to_pts': 128,
    'tdr_damping': 2,
}

ko = run(a,p,n,n_tdr,curve_opt_params)

In [ ]:
save_all(ko)